In [ ]:
# First, let's install Gradio if it's not already installed.
!pip install -q gradio

import gradio as gr
import random

# Placeholder for your DealAgentFramework or agent logic
# In a real scenario, you would import and initialize your agent here.
class PriceIsRightAgent:
    def __init__(self):
        # Initialize any agent-specific components, e.g., models, strategies
        pass

    def bid(self, item_description: str, current_highest_bid: float) -> float:
        """Simulates the agent making a bid based on the item and current bids."""
        print(f"Agent considering item: {item_description}, highest bid: {current_highest_bid}")
        # This is a very simple placeholder logic.
        # In a real agent, this would involve complex reasoning, knowledge retrieval, etc.
        if current_highest_bid < 1000:
            # Make a slightly higher bid, or a random bid within a range
            return round(current_highest_bid + random.uniform(10, 100), 2)
        else:
            # If the bid is already high, be more conservative or pass
            return round(current_highest_bid * 1.01, 2)

    def decide_if_highest(self, item_description: str, agent_bid: float, all_bids: list) -> str:
        """Simulates the agent determining if its bid is the highest without going over.
           This is a simplification; actual Price Is Right involves comparing to the actual price."""
        highest_valid_bid = 0.0
        for bid in all_bids:
            if bid <= actual_price: # Assume actual_price is known for this part, or is inferred.
                highest_valid_bid = max(highest_valid_bid, bid)

        if agent_bid == highest_valid_bid and agent_bid <= actual_price:
            return f"Agent wins with a bid of ${agent_bid}!"
        else:
            return f"Agent bid ${agent_bid}. Highest valid bid was ${highest_valid_bid}."

# Global instance of our placeholder agent
agent = PriceIsRightAgent()
actual_price = 0.0 # This would be set for each game instance

def price_is_right_game(item_description: str, user_bid: float, actual_item_price: float):
    global actual_price
    actual_price = actual_item_price

    # Agent makes its bid
    agent_bid_value = agent.bid(item_description, user_bid) # Agent bids after user

    # For simplicity, let's assume the game is just about comparing bids to the actual price
    # and not complex 'closest without over' logic for multiple players in this single turn.

    result_messages = []

    # User's outcome
    if user_bid <= actual_item_price and user_bid == actual_item_price:
        user_outcome = f"User bid ${user_bid}. Exact match! User wins!"
    elif user_bid <= actual_item_price:
        user_outcome = f"User bid ${user_bid}. Under actual price."
    else:
        user_outcome = f"User bid ${user_bid}. Over actual price!"
    result_messages.append(user_outcome)

    # Agent's outcome
    if agent_bid_value <= actual_item_price and agent_bid_value == actual_item_price:
        agent_outcome = f"Agent bid ${agent_bid_value}. Exact match! Agent wins!"
    elif agent_bid_value <= actual_item_price:
        agent_outcome = f"Agent bid ${agent_bid_value}. Under actual price."
    else:
        agent_outcome = f"Agent bid ${agent_bid_value}. Over actual price!"
    result_messages.append(agent_outcome)

    # Determine who is closer without going over
    closest_user = -float('inf')
    closest_agent = -float('inf')

    if user_bid <= actual_item_price:
        closest_user = user_bid
    if agent_bid_value <= actual_item_price:
        closest_agent = agent_bid_value

    winner = "No winner (both over or insufficient valid bids)"
    if closest_user > closest_agent:
        winner = "User is closer without going over!"
    elif closest_agent > closest_user:
        winner = "Agent is closer without going over!"
    elif closest_user > -float('inf') and closest_agent > -float('inf') and closest_user == closest_agent:
        winner = "Both user and agent are equally close without going over!"
    elif closest_user > -float('inf') or closest_agent > -float('inf'):
        winner = "One of them is valid, the other is not. Need more complex logic here."

    result_messages.append(f"The actual price was ${actual_item_price}.")
    result_messages.append(winner)

    return "\n".join(result_messages)

# Create the Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# Price-Is-Right Agent Game")
    gr.Markdown("Enter an item description, your bid, and the actual price to see how you and the agent fare.")

    item_input = gr.Textbox(label="Item Description", placeholder="e.g., Luxury Car")
    user_bid_input = gr.Number(label="Your Bid", value=100.0)
    actual_price_input = gr.Number(label="Actual Item Price (for simulation)", value=150.0)

    output_text = gr.Textbox(label="Game Results")

    play_button = gr.Button("Play Price Is Right!")
    play_button.click(price_is_right_game,
                      inputs=[item_input, user_bid_input, actual_price_input],
                      outputs=output_text)

demo.launch(debug=True)